# ===== G10 - CELL 1: SETUP & LOAD DATA =====

In [2]:
# ===== G10 - CELL 1: SETUP & LOAD DATA =====

import pandas as pd
import numpy as np
import warnings

from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")

# Same locked dataset used by G06 and G09
df = pd.read_csv("../output/G06_forecast_dataset.csv")

df["Month"] = pd.to_datetime(df["Month"])

print("=== G10 DATA CHECK ===")
print("Rows:", len(df))
print("Materials:", df["MaterialKey"].nunique())
print("Min month:", df["Month"].min())
print("Max month:", df["Month"].max())

print("\nTrain rows:",
      (df["IsTrainPeriod"] == 1).sum())

print("Test rows:",
      (df["IsTestPeriod"] == 1).sum())

print("\nTier distribution:")
print(
    df[["MaterialKey", "DSSEligibilityTier"]]
    .drop_duplicates()["DSSEligibilityTier"]
    .value_counts()
)

=== G10 DATA CHECK ===
Rows: 1145
Materials: 35
Min month: 2022-07-01 00:00:00
Max month: 2025-05-01 00:00:00

Train rows: 941
Test rows: 204

Tier distribution:
DSSEligibilityTier
A    28
B     7
Name: count, dtype: int64


#G10 - CELL 2: ARIMA ORDER SEARCH

In [4]:
# ===== G10 - CELL 2: ARIMA ORDER SEARCH =====

from itertools import product

# Bounded search space
ARIMA_ORDERS = list(
    product(
        range(3),   # p = 0, 1, 2
        range(2),   # d = 0, 1
        range(3)    # q = 0, 1, 2
    )
)

print("Candidate ARIMA orders:", len(ARIMA_ORDERS))
print(ARIMA_ORDERS)


def select_arima_order(train_values):
    """
    Select ARIMA(p,d,q) using the lowest AIC
    calculated from TRAIN data only.
    """

    y = np.asarray(train_values, dtype=float)

    best_order = None
    best_aic = np.inf
    best_fit = None

    successful_candidates = 0
    failed_candidates = 0

    for order in ARIMA_ORDERS:

        try:
            model = ARIMA(
                y,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted = model.fit()

            # Skip unusable AIC
            if not np.isfinite(fitted.aic):
                failed_candidates += 1
                continue

            successful_candidates += 1

            if fitted.aic < best_aic:
                best_aic = fitted.aic
                best_order = order
                best_fit = fitted

        except Exception:
            failed_candidates += 1

    if best_fit is None:
        raise RuntimeError(
            "No ARIMA candidate could be fitted."
        )

    return (
        best_order,
        best_aic,
        best_fit,
        successful_candidates,
        failed_candidates
    )


# ===== QUICK SMOKE TEST =====

sample_material = 5

sample_train = (
    df[
        (df["MaterialKey"] == sample_material)
        & (df["IsTrainPeriod"] == 1)
    ]
    .sort_values("Month")["RequirementQty_PU"]
    .to_numpy()
)

(
    best_order,
    best_aic,
    best_fit,
    successful_candidates,
    failed_candidates
) = select_arima_order(sample_train)

sample_forecast = np.asarray(
    best_fit.forecast(steps=6),
    dtype=float
)


print("\n=== ARIMA SMOKE TEST ===")

print("MaterialKey:", sample_material)
print("Train observations:", len(sample_train))
print("Best order:", best_order)
print("Best AIC:", round(best_aic, 4))
print("Successful candidates:", successful_candidates)
print("Failed candidates:", failed_candidates)

print(
    "Raw 6-month forecast:",
    np.round(sample_forecast, 2)
)

Candidate ARIMA orders: 18
[(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 0), (0, 1, 1), (0, 1, 2), (1, 0, 0), (1, 0, 1), (1, 0, 2), (1, 1, 0), (1, 1, 1), (1, 1, 2), (2, 0, 0), (2, 0, 1), (2, 0, 2), (2, 1, 0), (2, 1, 1), (2, 1, 2)]

=== ARIMA SMOKE TEST ===
MaterialKey: 5
Train observations: 29
Best order: (0, 1, 2)
Best AIC: 459.8681
Successful candidates: 18
Failed candidates: 0
Raw 6-month forecast: [3930.2  4079.92 4079.92 4079.92 4079.92 4079.92]


Cell 3: RUN ARIMA BACKTEST FOR ALL MATERIALS.

In [5]:
# ===== G10 - CELL 3: RUN ARIMA BACKTEST FOR ALL MATERIALS =====

# Shared metrics - same definitions as G06/G09
def mae(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return np.mean(np.abs(forecast - actual))


def rmse(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return np.sqrt(np.mean((forecast - actual) ** 2))


def bias(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return np.mean(forecast - actual)


arima_predictions = []
arima_metrics = []
arima_logs = []


for material_key, group in df.groupby("MaterialKey"):

    group = group.sort_values("Month")

    train = group[
        group["IsTrainPeriod"] == 1
    ].copy()

    test = group[
        group["IsTestPeriod"] == 1
    ].copy()

    train_values = (
        train["RequirementQty_PU"]
        .astype(float)
        .to_numpy()
    )

    # -----------------------------------------
    # Cannot evaluate without complete test set
    # -----------------------------------------
    if len(test) != 6:

        status = (
            f"INSUFFICIENT_TEST_DATA_{len(test)}_OF_6"
        )

        arima_metrics.append({
            "MaterialKey": material_key,
            "Model": "ARIMA",
            "Order": None,
            "AIC": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
            "Bias": np.nan,
            "Status": status
        })

        arima_logs.append({
            "MaterialKey": material_key,
            "TrainObserved": len(train),
            "TestObserved": len(test),
            "SelectedOrder": None,
            "AIC": np.nan,
            "SuccessfulCandidates": np.nan,
            "FailedCandidates": np.nan,
            "NegativeForecastsRaw": np.nan,
            "Status": status,
            "Message": ""
        })

        continue

    try:

        # -----------------------------------------
        # Select order using TRAIN AIC only
        # -----------------------------------------
        (
            best_order,
            best_aic,
            best_fit,
            successful_candidates,
            failed_candidates
        ) = select_arima_order(train_values)

        # -----------------------------------------
        # 6-month fixed-origin forecast
        # -----------------------------------------
        raw_forecast = np.asarray(
            best_fit.forecast(steps=6),
            dtype=float
        )

        negative_count = int(
            (raw_forecast < 0).sum()
        )

        # Business constraint:
        # material requirement cannot be negative
        forecast_values = np.maximum(
            raw_forecast,
            0
        )

        actual_values = (
            test["RequirementQty_PU"]
            .astype(float)
            .to_numpy()
        )

        # -----------------------------------------
        # Store monthly predictions
        # -----------------------------------------
        for month, actual, raw_f, forecast in zip(
            test["Month"],
            actual_values,
            raw_forecast,
            forecast_values
        ):

            arima_predictions.append({
                "MaterialKey": material_key,
                "ForecastMonth": month,
                "Model": "ARIMA",
                "Order": str(best_order),
                "ActualQty": actual,
                "RawForecastQty": raw_f,
                "ForecastQty": forecast,
                "WasClipped": int(raw_f < 0),
                "Error": forecast - actual
            })

        # -----------------------------------------
        # Store evaluation metrics
        # -----------------------------------------
        arima_metrics.append({
            "MaterialKey": material_key,
            "Model": "ARIMA",
            "Order": str(best_order),
            "AIC": best_aic,
            "MAE": mae(
                actual_values,
                forecast_values
            ),
            "RMSE": rmse(
                actual_values,
                forecast_values
            ),
            "Bias": bias(
                actual_values,
                forecast_values
            ),
            "Status": "SUCCESS"
        })

        # -----------------------------------------
        # Store model-selection log
        # -----------------------------------------
        arima_logs.append({
            "MaterialKey": material_key,
            "TrainObserved": len(train),
            "TestObserved": len(test),
            "SelectedOrder": str(best_order),
            "AIC": best_aic,
            "SuccessfulCandidates":
                successful_candidates,
            "FailedCandidates":
                failed_candidates,
            "NegativeForecastsRaw":
                negative_count,
            "Status": "SUCCESS",
            "Message": (
                "Negative forecasts clipped to 0"
                if negative_count > 0
                else ""
            )
        })

    except Exception as e:

        status = "FIT_FAILED"

        arima_metrics.append({
            "MaterialKey": material_key,
            "Model": "ARIMA",
            "Order": None,
            "AIC": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
            "Bias": np.nan,
            "Status": status
        })

        arima_logs.append({
            "MaterialKey": material_key,
            "TrainObserved": len(train),
            "TestObserved": len(test),
            "SelectedOrder": None,
            "AIC": np.nan,
            "SuccessfulCandidates": 0,
            "FailedCandidates": 18,
            "NegativeForecastsRaw": np.nan,
            "Status": status,
            "Message":
                f"{type(e).__name__}: {str(e)}"
        })


arima_predictions = pd.DataFrame(
    arima_predictions
)

arima_metrics = pd.DataFrame(
    arima_metrics
)

arima_logs = pd.DataFrame(
    arima_logs
)


# ===== SUMMARY =====

print("=== ARIMA BATCH RESULTS ===")

print("\nPrediction rows:")
print(len(arima_predictions))

print("\nMetric rows:")
print(len(arima_metrics))

print("\nStatus:")
print(
    arima_metrics["Status"]
    .value_counts()
)

print("\nRaw negative forecasts:")
print(
    arima_logs["NegativeForecastsRaw"]
    .fillna(0)
    .sum()
)

print("\nFit failures:")
print(
    (arima_metrics["Status"] == "FIT_FAILED")
    .sum()
)

=== ARIMA BATCH RESULTS ===

Prediction rows:
204

Metric rows:
35

Status:
Status
SUCCESS                          34
INSUFFICIENT_TEST_DATA_0_OF_6     1
Name: count, dtype: int64

Raw negative forecasts:
0.0

Fit failures:
0


Cell 4: VALIDATE ARIMA RESULTS.

In [6]:
# ===== G10 - CELL 4: VALIDATE ARIMA RESULTS =====

print("=== ARIMA QA ===")

print("Prediction rows:",
      len(arima_predictions))

print(
    "Materials with predictions:",
    arima_predictions["MaterialKey"].nunique()
)

print(
    "Duplicate Material-Month-Model:",
    arima_predictions.duplicated(
        subset=[
            "MaterialKey",
            "ForecastMonth",
            "Model"
        ]
    ).sum()
)

print(
    "NULL Actual:",
    arima_predictions["ActualQty"].isna().sum()
)

print(
    "NULL Forecast:",
    arima_predictions["ForecastQty"].isna().sum()
)

print(
    "Negative final forecasts:",
    (arima_predictions["ForecastQty"] < 0).sum()
)


# ===== ORDER DISTRIBUTION =====

successful_arima = arima_metrics[
    arima_metrics["Status"] == "SUCCESS"
].copy()

print("\n=== SELECTED ARIMA ORDERS ===")

print(
    successful_arima["Order"]
    .value_counts()
)


# ===== CANDIDATE SEARCH QA =====

print("\n=== ARIMA SEARCH QA ===")

print(
    "Total successful candidate fits:",
    arima_logs["SuccessfulCandidates"]
    .fillna(0)
    .sum()
)

print(
    "Total failed candidate fits:",
    arima_logs["FailedCandidates"]
    .fillna(0)
    .sum()
)


# ===== NON-SUCCESS MATERIALS =====

print("\n=== NON-SUCCESS MATERIAL ===")

display(
    arima_metrics[
        arima_metrics["Status"] != "SUCCESS"
    ]
)


# ===== SAMPLE RESULTS =====

print("\n=== SAMPLE ARIMA METRICS ===")

display(
    successful_arima[
        [
            "MaterialKey",
            "Order",
            "AIC",
            "MAE",
            "RMSE",
            "Bias"
        ]
    ].head(10)
)

=== ARIMA QA ===
Prediction rows: 204
Materials with predictions: 34
Duplicate Material-Month-Model: 0
NULL Actual: 0
NULL Forecast: 0
Negative final forecasts: 0

=== SELECTED ARIMA ORDERS ===
Order
(2, 1, 2)    28
(0, 1, 2)     6
Name: count, dtype: int64

=== ARIMA SEARCH QA ===
Total successful candidate fits: 612.0
Total failed candidate fits: 0.0

=== NON-SUCCESS MATERIAL ===


,MaterialKey,Model,Order,AIC,MAE,RMSE,Bias,Status
22,169,ARIMA,NaN,NaN,NaN,NaN,NaN,INSUFFICIENT_TEST_DATA_0_OF_6



=== SAMPLE ARIMA METRICS ===


,MaterialKey,Order,AIC,MAE,RMSE,Bias
0,4,"(2, 1, 2)",19.857646,863.600110,1207.014960,269.441546
1,5,"(0, 1, 2)",459.868078,1448.713647,1791.669011,213.296060
2,6,"(0, 1, 2)",402.818228,516.892097,618.968348,153.486499
3,9,"(2, 1, 2)",356.333029,324.658671,473.270603,-22.469010
4,11,"(2, 1, 2)",478.835272,3231.972906,4606.233684,1457.415868
5,12,"(2, 1, 2)",444.611280,1363.526193,2186.251428,294.742064
6,13,"(2, 1, 2)",425.766220,1328.611077,1919.735536,-123.699271
7,23,"(2, 1, 2)",425.503978,1291.546564,1882.682367,-98.423531
8,24,"(2, 1, 2)",425.503978,1291.546564,1882.682367,-98.423531
9,25,"(2, 1, 2)",424.755596,1257.047987,1847.149342,-75.665671


Cell 5: EXPORT ARIMA RESULTS

In [7]:
# ===== G10 - CELL 5: EXPORT ARIMA RESULTS =====

arima_predictions = arima_predictions.sort_values(
    ["MaterialKey", "ForecastMonth"]
).reset_index(drop=True)

arima_metrics = arima_metrics.sort_values(
    ["MaterialKey"]
).reset_index(drop=True)

arima_logs = arima_logs.sort_values(
    ["MaterialKey"]
).reset_index(drop=True)


arima_predictions.to_csv(
    "../output/G10_arima_predictions.csv",
    index=False
)

arima_metrics.to_csv(
    "../output/G10_arima_metrics.csv",
    index=False
)

arima_logs.to_csv(
    "../output/G10_arima_logs.csv",
    index=False
)


print("=== G10 EXPORT COMPLETED ===")

print(
    "Prediction rows:",
    len(arima_predictions)
)

print(
    "Metric rows:",
    len(arima_metrics)
)

print(
    "Log rows:",
    len(arima_logs)
)

print(
    "Successful evaluations:",
    (arima_metrics["Status"] == "SUCCESS").sum()
)

print(
    "Fit failures:",
    (arima_metrics["Status"] == "FIT_FAILED").sum()
)

print(
    "Negative forecasts:",
    arima_predictions["WasClipped"].sum()
)

print(
    "Failed candidate fits:",
    arima_logs["FailedCandidates"]
    .fillna(0)
    .sum()
)

=== G10 EXPORT COMPLETED ===
Prediction rows: 204
Metric rows: 35
Log rows: 35
Successful evaluations: 34
Fit failures: 0
Negative forecasts: 0
Failed candidate fits: 0.0
